In [1]:
# Environment Paths & Directory Setup
import logging
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    has_colab = True
except Exception:
    has_colab = False

# Path Resolution
DEFAULT_COLAB_ROOT = Path("/content/drive/MyDrive/MPLADs")
ALT_COLAB_ROOT = Path("/content/drive/MYDrive/MPLADs")

if has_colab and DEFAULT_COLAB_ROOT.exists():
    PROJECT_ROOT = DEFAULT_COLAB_ROOT
elif has_colab and ALT_COLAB_ROOT.exists():
    PROJECT_ROOT = ALT_COLAB_ROOT
elif (Path.cwd() / "shared_preprocessing_artifacts").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd()

SHARED_ARTIFACT_DIR = PROJECT_ROOT / "shared_preprocessing_artifacts"
FEATURE3_ARTIFACT_DIR = PROJECT_ROOT / "feature3_artifacts"
FEATURE1_ARTIFACT_DIR = PROJECT_ROOT / "feature1_artifacts"
OUTPUT_DIR = PROJECT_ROOT / "feature2_artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger("mplads_feature2")

logger.info("Project Root: %s", PROJECT_ROOT)
logger.info("Shared Artifacts Dir: %s", SHARED_ARTIFACT_DIR)
logger.info("Feature 3 Artifacts Dir: %s", FEATURE3_ARTIFACT_DIR)
logger.info("Feature 1 Artifacts Dir: %s", FEATURE1_ARTIFACT_DIR)
logger.info("Feature 2 Output Dir: %s", OUTPUT_DIR)



Mounted at /content/drive


In [2]:
# Data Ingestion & Data Bus Resolution
import pandas as pd
import numpy as np

def read_artifact(dir_path: Path, name: str) -> pd.DataFrame:
    """Reads parquet artifact if available, falling back to CSV."""
    parquet_path = dir_path / f"{name}.parquet"
    csv_path = dir_path / f"{name}.csv"

    if parquet_path.exists():
        try:
            logger.info("Loading parquet: %s", parquet_path)
            return pd.read_parquet(parquet_path)
        except Exception as e:
            logger.warning("Failed loading parquet %s (%s). Falling back to CSV.", parquet_path, e)

    if csv_path.exists():
        logger.info("Loading CSV: %s", csv_path)
        return pd.read_csv(csv_path, low_memory=False)

    raise FileNotFoundError(f"Could not find {name}.parquet or {name}.csv in {dir_path}")

# Ingestion Hierarchy:
# 1. feature3_artifacts/feature3_fact_work_enriched
# 2. shared_preprocessing_artifacts/fact_work_feature1
# 3. feature1_artifacts/feature1_fact_work_disbursement_risk
# 4. shared_preprocessing_artifacts/fact_work
work_df = None
source_name = ""

for dir_path, name in [
    (FEATURE3_ARTIFACT_DIR, "feature3_fact_work_enriched"),
    (SHARED_ARTIFACT_DIR, "fact_work_feature1"),
    (FEATURE1_ARTIFACT_DIR, "feature1_fact_work_disbursement_risk"),
    (SHARED_ARTIFACT_DIR, "fact_work"),
]:
    try:
        work_df = read_artifact(dir_path, name)
        source_name = f"{dir_path.name}/{name}"
        logger.info("Successfully ingested input data from %s (shape: %s)", source_name, work_df.shape)
        break
    except FileNotFoundError:
        continue

if work_df is None:
    raise FileNotFoundError("Could not locate any valid fact_work input dataset across shared and feature artifact directories.")

# Standardize key column
key_col = "work_key" if "work_key" in work_df.columns else "work_id"
if key_col not in work_df.columns:
    raise KeyError("Neither 'work_key' nor 'work_id' found in input dataset.")
if "work_key" not in work_df.columns:
    work_df["work_key"] = work_df[key_col]
if "work_id" not in work_df.columns:
    work_df["work_id"] = work_df[key_col]

logger.info("Dataset key column: %s | Total records: %d", key_col, len(work_df))



In [3]:
# Feature Engineering & Preprocessing
df = work_df.copy()

# 1. Clean Sanction Amount & Log Transformation
df["sanction_amount"] = pd.to_numeric(df["sanction_amount"], errors="coerce")
df["valid_cost_mask"] = df["sanction_amount"].notna() & (df["sanction_amount"] > 0)

# Log transform log(1 + sanction_amount)
df["log_sanction_amount"] = np.where(
    df["valid_cost_mask"],
    np.log1p(df["sanction_amount"]),
    np.nan
)

# 2. State & Category Grouping Keys
df["state_key_clean"] = df.get("state_key", df.get("state", pd.Series("UNKNOWN_STATE", index=df.index))).fillna("UNKNOWN_STATE")
df["category_nlp_clean"] = df.get("category_nlp", pd.Series("Unclassified", index=df.index)).fillna("Unclassified")

df["peer_group_key"] = df["state_key_clean"].astype(str) + " | " + df["category_nlp_clean"].astype(str)
df["cat_group_key"] = df["category_nlp_clean"].astype(str)

# 3. Work Description Length & Token Count
desc_col = next((c for c in ["work_description_clean", "nlp_description", "work_description_raw"] if c in df.columns), None)
desc_series = df[desc_col].fillna("").astype(str) if desc_col else pd.Series("", index=df.index)
df["desc_length_chars"] = desc_series.str.len()
df["desc_token_count"] = desc_series.str.split().str.len()
assert df["desc_token_count"].max() > 0, (
    "desc_token_count is degenerate (max=0). "
    "Input artifact lacks description content. "
    f"Resolved column: {desc_col}. "
    f"Non-empty descriptions: {(df['desc_token_count'] > 0).sum()} / {len(df)}"
)


# 4. MP Recommendation Intensity / Frequency within Category
mp_col = "mp_key" if "mp_key" in df.columns else ("mp" if "mp" in df.columns else None)
if mp_col and mp_col in df.columns:
    df["mp_key_clean"] = df[mp_col].fillna("UNKNOWN_MP")
    mp_cat_counts = df.groupby(["mp_key_clean", "category_nlp_clean"])["work_key"].transform("count")
    df["mp_work_count_category"] = mp_cat_counts
else:
    df["mp_work_count_category"] = 1

df["log_mp_work_count_category"] = np.log1p(df["mp_work_count_category"])

logger.info("Preprocessing complete.")
logger.info("Valid cost records: %d / %d (%.1f%%)", df["valid_cost_mask"].sum(), len(df), df["valid_cost_mask"].mean() * 100)
logger.info("Log sanction amount range: [%.2f, %.2f]", df["log_sanction_amount"].min(), df["log_sanction_amount"].max())



In [4]:
# 1. State + Category Peer Group Statistics (Fitted on Training Partition Only)
train_valid_df = df[df["valid_cost_mask"] & (df["model_split"] == "training")].copy()

def calc_mad(x):
    if len(x) == 0:
        return 0.05
    med = np.median(x)
    return float(np.median(np.abs(x - med)))

peer_stats = train_valid_df.groupby("peer_group_key").agg(
    peer_group_size=("log_sanction_amount", "count"),
    peer_median_log_amount=("log_sanction_amount", "median"),
    peer_mad_log_amount=("log_sanction_amount", calc_mad),
    peer_median_amount=("sanction_amount", "median")
).reset_index()

cat_stats = train_valid_df.groupby("cat_group_key").agg(
    cat_group_size=("log_sanction_amount", "count"),
    cat_median_log_amount=("log_sanction_amount", "median"),
    cat_mad_log_amount=("log_sanction_amount", calc_mad),
    cat_median_amount=("sanction_amount", "median")
).reset_index()

df = df.merge(peer_stats, on="peer_group_key", how="left")
df = df.merge(cat_stats, on="cat_group_key", how="left")

category_conf = df.get("category_confidence", df.get("category_nlp_confidence", pd.Series(1.0, index=df.index)))
use_state_cat = (df.get("category_agreement_count", pd.Series(0, index=df.index)) >= 2) & (~df.get("category_is_noise", pd.Series(False, index=df.index)).fillna(False)) & (category_conf >= 0.55)

eff_med_log = np.where(use_state_cat & (df["peer_group_size"].fillna(0) >= 10), df["peer_median_log_amount"], df["cat_median_log_amount"].fillna(0.0))
eff_mad_log = np.where(use_state_cat & (df["peer_group_size"].fillna(0) >= 10), df["peer_mad_log_amount"], df["cat_mad_log_amount"].fillna(0.05))
eff_med_amt = np.where(use_state_cat & (df["peer_group_size"].fillna(0) >= 10), df["peer_median_amount"], df["cat_median_amount"].fillna(100000.0))

mad_log_denominator = np.maximum(1.4826 * np.nan_to_num(eff_mad_log, nan=0.05), 0.05)

df["peer_cost_zscore"] = np.where(
    df["valid_cost_mask"],
    (df["log_sanction_amount"] - eff_med_log) / mad_log_denominator,
    0.0
)

df["upper_cost_zscore"] = np.maximum(0.0, df["peer_cost_zscore"])
df["sanction_peer_ratio"] = np.where(df["valid_cost_mask"] & (eff_med_amt > 0), df["sanction_amount"] / eff_med_amt, 1.0)
df["effective_peer_median_amount"] = eff_med_amt

df["peer_reference_version"] = "v2.0_training_partition"
df["peer_reference_level"] = np.where(use_state_cat & (df["peer_group_size"].fillna(0) >= 10), "state_category", "category_national")
df["peer_reference_count"] = np.where(df["peer_reference_level"] == "state_category", df["peer_group_size"].fillna(0), df["cat_group_size"].fillna(0))

logger.info("Peer statistics fitted on training partition. Model splits: %s", df["model_split"].value_counts().to_dict())




In [5]:
# Multivariate Local Outlier Factor (LOF) Anomaly Detection (Fitted on Training Partition Only)
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
import joblib

logger.info("--- Starting Multivariate Local Outlier Factor (LOF) Pipeline ---")

lof_feature_cols = [
    "log_sanction_amount",
    "desc_token_count",
    "upper_cost_zscore",
    "log_mp_work_count_category"
]

train_mask = df["valid_cost_mask"] & (df["model_split"] == "training")
X_raw_train = df.loc[train_mask, lof_feature_cols].copy()

imputer = SimpleImputer(strategy="median")
scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(imputer.fit_transform(X_raw_train))

lof_model = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.03,
    novelty=True,
    n_jobs=-1
)
unique_X_train_scaled = np.unique(X_train_scaled, axis=0)
lof_model.fit(unique_X_train_scaled)

valid_cost_indices = df[df["valid_cost_mask"]].index
X_raw_all = df.loc[valid_cost_indices, lof_feature_cols].copy()
X_all_scaled = scaler.transform(imputer.transform(X_raw_all))

raw_lof_scores = -lof_model.score_samples(X_all_scaled)

train_raw_scores = raw_lof_scores[df.loc[valid_cost_indices, "model_split"] == "training"]
q1, q99 = np.percentile(train_raw_scores, [1, 99])
norm_lof_scores = np.clip((raw_lof_scores - q1) / (q99 - q1 if q99 > q1 else 1.0), 0.0, 1.0)
assert q99 < 1000, (
    f"LOF q99 is degenerate ({q99:.2f}). "
    "Check desc_token_count and other LOF features for constant values."
)


df["lof_anomaly_score"] = np.nan
df.loc[valid_cost_indices, "lof_anomaly_score"] = norm_lof_scores

lof_model_path = OUTPUT_DIR / "feature2_lof_model.joblib"
joblib.dump({
    "imputer": imputer,
    "scaler": scaler,
    "model": lof_model,
    "feature_cols": lof_feature_cols,
    "q1": float(q1),
    "q99": float(q99)
}, lof_model_path)

logger.info("LOF model fitted on %d training records and applied across full dataset.", train_mask.sum())




In [6]:
# Cost Anomaly Rules & Unified Blended Cost Risk Score (Candidate C: Component Weights Sum = 1.0)
df["is_vague_high_cost"] = df["valid_cost_mask"] & (df["sanction_amount"] >= 1000000.0) & (df["desc_token_count"] < 5)
df["is_extreme_peer_overrun"] = df["valid_cost_mask"] & (df["sanction_peer_ratio"] >= 3.0)

z_norm = np.clip(df["upper_cost_zscore"] / 4.0, 0.0, 1.0)
ratio_norm = np.clip((df["sanction_peer_ratio"] - 1.0) / 4.0, 0.0, 1.0)
lof_norm = df["lof_anomaly_score"].fillna(0.0)

vague_desc_boost = np.where(df["is_vague_high_cost"], 0.10, 0.0)

# Validated Candidate C: Weights sum to 1.0 (40% Z-Score + 35% LOF Anomaly + 25% Peer Ratio + Vague Boost capped to 1.0)
df["cost_risk_score"] = np.where(
    df["valid_cost_mask"],
    np.clip(0.40 * z_norm + 0.35 * lof_norm + 0.25 * ratio_norm + vague_desc_boost, 0.0, 1.0),
    np.nan
)

def assign_cost_risk_tier(row: pd.Series) -> str:
    if not row["valid_cost_mask"] or pd.isna(row["cost_risk_score"]):
        return "Unknown / Invalid"

    score = row["cost_risk_score"]
    zscore = row["upper_cost_zscore"]
    ratio = row["sanction_peer_ratio"]

    if score >= 0.70 or (zscore >= 4.0 and ratio >= 3.0):
        return "Critical"
    elif score >= 0.50 or zscore >= 2.5:
        return "High"
    elif score >= 0.30:
        return "Medium"
    else:
        return "Low"

df["cost_risk_tier"] = df.apply(assign_cost_risk_tier, axis=1)

row_level_peer_cols = [
    "effective_peer_median_amount", "peer_cost_zscore", "upper_cost_zscore",
    "sanction_peer_ratio", "lof_anomaly_score", "cost_risk_score",
]
for col in row_level_peer_cols:
    if col in df.columns:
        df.loc[~df["valid_cost_mask"], col] = np.nan


print("--- Feature 2 Cost Risk Tier Distribution ---")
print(df["cost_risk_tier"].value_counts(dropna=False))




--- Feature 2 Cost Risk Tier Distribution ---
cost_risk_tier
Low                  28657
Unknown / Invalid     9150
Medium                2751
High                  1760
Critical               832
Name: count, dtype: int64


In [7]:
# Dashboard Explanation Generator

def generate_cost_dashboard_explanation(row: pd.Series) -> str:
    """Generates clear, natural language explanation strings for cost anomaly dashboard display."""
    if not row["valid_cost_mask"]:
        return "Cost status unknown: Missing or non-positive sanction amount recorded."

    sanction = row["sanction_amount"]
    peer_median = row["effective_peer_median_amount"]
    ratio = row["sanction_peer_ratio"]
    zscore = row["upper_cost_zscore"]
    lof_score = row.get("lof_anomaly_score", np.nan)
    state = row["state_key_clean"]
    cat = row["category_nlp_clean"]
    tier = row["cost_risk_tier"]

    z_val = zscore if pd.notna(zscore) else 0.0
    z_str = f">{10.0:.1f} sigma" if z_val >= 10.0 else f"{z_val:.1f} sigma"
    lof_note = f" (Multivariate LOF anomaly score: {lof_score:.2f})" if pd.notna(lof_score) and lof_score >= 0.60 else ""

    if tier == "Critical":
        return (
            f"CRITICAL COST ANOMALY: Sanctioned amount Rs.{sanction:,.0f} is {ratio:.1f}x higher than the "
            f"{state} ({cat}) peer median (Rs.{peer_median:,.0f}). Exceeds expected peer median by {z_str}.{lof_note}"
        )
    elif tier == "High":
        return (
            f"HIGH COST OVERRUN: Sanctioned amount Rs.{sanction:,.0f} exceeds {state} ({cat}) "
            f"peer median of Rs.{peer_median:,.0f} by {z_str} ({ratio:.1f}x peer benchmark).{lof_note}"
        )
    elif pd.notna(lof_score) and lof_score >= 0.70 and zscore < 2.0:
        return (
            f"MULTIVARIATE COST ANOMALY DETECTED: Project displays unusual combination of cost, description length, "
            f"and MP recommendation frequency (LOF Anomaly Score: {lof_score:.2f}) despite moderate univariate cost z-score."
        )
    elif tier == "Medium":
        return (
            f"MODERATE COST VARIANCE: Sanctioned amount Rs.{sanction:,.0f} is {ratio:.1f}x relative to "
            f"peer group median Rs.{peer_median:,.0f} ({z_str} above peer median).{lof_note}"
        )
    else:
        return (
            f"NORMAL COST BENCHMARK: Sanctioned amount Rs.{sanction:,.0f} is within normal expectation for "
            f"{state} ({cat}) (Peer median: Rs.{peer_median:,.0f})."
        )

df["cost_risk_explanation"] = df.apply(generate_cost_dashboard_explanation, axis=1)

print("--- Sample Critical Cost Dashboard Explanations ---")
critical_samples = df[df["cost_risk_tier"] == "Critical"]["cost_risk_explanation"].head(5)
for i, exp in enumerate(critical_samples, 1):
    print(f"{i}. {exp}\n")



--- Sample Critical Cost Dashboard Explanations ---
1. CRITICAL COST ANOMALY: Sanctioned amount Rs.37,908,000 is 163.3x higher than the Uttar Pradesh (Street Lighting & Solar Energy) peer median (Rs.232,198). Exceeds expected peer median by 6.8 sigma. (Multivariate LOF anomaly score: 0.71)

2. CRITICAL COST ANOMALY: Sanctioned amount Rs.5,925,309 is 11.9x higher than the West Bengal (Road & Pathway Infrastructure) peer median (Rs.500,000). Exceeds expected peer median by 2.7 sigma. (Multivariate LOF anomaly score: 0.61)

3. CRITICAL COST ANOMALY: Sanctioned amount Rs.9,511,630 is 10.1x higher than the Uttar Pradesh (Road & Pathway Infrastructure) peer median (Rs.943,851). Exceeds expected peer median by 4.7 sigma.

4. CRITICAL COST ANOMALY: Sanctioned amount Rs.150,000 is 6.8x higher than the Jharkhand (Street Lighting & Solar Energy) peer median (Rs.22,000). Exceeds expected peer median by >10.0 sigma.

5. CRITICAL COST ANOMALY: Sanctioned amount Rs.150,000 is 6.8x higher than the Jha

In [8]:
# Summary Analysis & Top Anomaly Worklist Display

display_cols = [
    "work_id", "state_key_clean", "category_nlp_clean", "sanction_amount",
    "effective_peer_median_amount", "sanction_peer_ratio", "peer_cost_zscore",
    "lof_anomaly_score", "cost_risk_score", "cost_risk_tier", "cost_risk_explanation"
]

critical_cost_worklist = df[df["cost_risk_tier"] == "Critical"].sort_values(
    by=["cost_risk_score", "sanction_peer_ratio"],
    ascending=[False, False]
)

print(f"Total Critical Cost Anomalies Identified: {len(critical_cost_worklist)}")
print("\n--- Top 10 Critical Sanction-Amount Cost Anomalies ---")
if len(critical_cost_worklist) > 0:
    display_df = critical_cost_worklist[display_cols].head(10)
    try:
        from IPython.display import display
        display(display_df)
    except Exception:
        print(display_df.to_string())



Total Critical Cost Anomalies Identified: 832

--- Top 10 Critical Sanction-Amount Cost Anomalies ---


,work_id,state_key_clean,category_nlp_clean,sanction_amount,effective_peer_median_amount,sanction_peer_ratio,peer_cost_zscore,lof_anomaly_score,cost_risk_score,cost_risk_tier,cost_risk_explanation
27581,WS/MP176/2024-2025/149419,Chhattisgarh,School & Education Infrastructure,30000000.0,100000.00,300.000000,114.075450,1.0,1.0,Critical,CRITICAL COST ANOMALY: Sanctioned amount Rs.30...
31944,WS/MP18385/2025-2026/198850,Jharkhand,Street Lighting & Solar Energy,4969990.0,22000.00,225.908636,108.401708,1.0,1.0,Critical,"CRITICAL COST ANOMALY: Sanctioned amount Rs.4,..."
2927,WS/MP18201/2024-2025/141045,Uttar Pradesh,Street Lighting & Solar Energy,46470400.0,216400.00,214.743068,13.497589,1.0,1.0,Critical,CRITICAL COST ANOMALY: Sanctioned amount Rs.46...
27285,WS/MP18363/2024-2025/147448,Karnataka,Healthcare & Ambulance Services,16548000.0,107771.00,153.547800,100.680054,1.0,1.0,Critical,CRITICAL COST ANOMALY: Sanctioned amount Rs.16...
32140,WS/MP18385/2025-2026/200585,Jharkhand,Street Lighting & Solar Energy,3234607.0,22000.00,147.027591,99.811502,1.0,1.0,Critical,"CRITICAL COST ANOMALY: Sanctioned amount Rs.3,..."
31312,WS/MP187/2025-2026/194547,Uttar Pradesh,Street Lighting & Solar Energy,29889000.0,216400.00,138.119224,12.388195,1.0,1.0,Critical,CRITICAL COST ANOMALY: Sanctioned amount Rs.29...
18812,WS/MP507/2024-2025/170558,Uttar Pradesh,Street Lighting & Solar Energy,28310620.0,216400.00,130.825416,12.251813,1.0,1.0,Critical,CRITICAL COST ANOMALY: Sanctioned amount Rs.28...
30829,WS/MP839/2025-2026/191381,Jammu And Kashmir,Street Lighting & Solar Energy,21870000.0,243000.00,90.000000,89.996112,1.0,1.0,Critical,CRITICAL COST ANOMALY: Sanctioned amount Rs.21...
27283,WS/MP18363/2024-2025/147435,Karnataka,Healthcare & Ambulance Services,8580000.0,107771.00,79.613254,87.543428,1.0,1.0,Critical,"CRITICAL COST ANOMALY: Sanctioned amount Rs.8,..."
27476,WS/MP197/2024-2025/149134,Uttar Pradesh,Street Lighting & Solar Energy,12150000.0,232197.98,52.326037,5.316150,1.0,1.0,Critical,CRITICAL COST ANOMALY: Sanctioned amount Rs.12...


In [9]:
# Artifact Export & Metadata
import json
from datetime import datetime

# Export Canonical Fact Table
shared_parquet_path = SHARED_ARTIFACT_DIR / "fact_work_feature2.parquet"
shared_csv_path = SHARED_ARTIFACT_DIR / "fact_work_feature2.csv"

df.to_csv(shared_csv_path, index=False)
try:
    df.to_parquet(shared_parquet_path, index=False)
except Exception as e:
    logger.warning("Failed saving parquet: %s", e)

# Export Feature 2 Artifacts
feat2_csv_path = OUTPUT_DIR / "feature2_fact_work_cost_risk.csv"
feat2_parquet_path = OUTPUT_DIR / "feature2_fact_work_cost_risk.parquet"

df.to_csv(feat2_csv_path, index=False)
try:
    df.to_parquet(feat2_parquet_path, index=False)
except Exception:
    pass

# Export Complete Validation Metadata
tier_counts = df["cost_risk_tier"].value_counts().to_dict()
metadata = {
    "feature": "Feature 2 - Cost Benchmarking & Sanction-Amount Anomaly Detection",
    "run_timestamp": datetime.now().isoformat(),
    "split_method": "chronological_date_split",
    "date_column": "obs_date",
    "train_date_min": str(df.loc[df["model_split"] == "training", "obs_date"].min()),
    "train_date_max": str(df.loc[df["model_split"] == "training", "obs_date"].max()),
    "validation_date_min": str(df.loc[df["model_split"] == "validation", "obs_date"].min()),
    "validation_date_max": str(df.loc[df["model_split"] == "validation", "obs_date"].max()),
    "holdout_date_min": str(df.loc[df["model_split"] == "holdout", "obs_date"].min()),
    "holdout_date_max": str(df.loc[df["model_split"] == "holdout", "obs_date"].max()),
    "selected_weights": {"z_score": 0.40, "lof_anomaly": 0.35, "peer_ratio": 0.25},
    "vague_boost_value": 0.10,
    "normalization_method": "percentile_q1_q99",
    "normalization_quantiles": {"q1": float(q1), "q99": float(q99)},
    "risk_tier_counts": {str(k): int(v) for k, v in tier_counts.items()}
}

metadata_path = OUTPUT_DIR / "feature2_run_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
logger.info("Saved metadata JSON to %s", metadata_path)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plot_df = df[df["valid_cost_mask"]].copy()

# Plot 1: Actual Cost vs Peer Median Cost (Log-Log Parity Plot)
fig1, ax1 = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=plot_df,
    x="effective_peer_median_amount",
    y="sanction_amount",
    hue="cost_risk_tier",
    hue_order=["Low", "Medium", "High", "Critical"],
    palette={"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e67e22", "Critical": "#e74c3c"},
    alpha=0.7,
    s=35,
    ax=ax1
)

x_vals = np.logspace(4, 7, 100)
ax1.plot(x_vals, x_vals, 'k--', alpha=0.6, label="1:1 Equal Cost Line (Normal)")
ax1.plot(x_vals, 3 * x_vals, 'r--', alpha=0.6, label="3x Overrun Line (Critical Threshold)")

ax1.set_xscale("log")
ax1.set_yscale("log")
ax1.set_xlabel("Peer Group Median Sanction Amount (Rs., Log Scale)", fontsize=11)
ax1.set_ylabel("Actual Sanction Amount (Rs., Log Scale)", fontsize=11)
ax1.set_title("Feature 2: Actual Cost vs. Peer Median Cost (Outliers Highlighted)", fontsize=13, fontweight="bold")
ax1.legend(title="Cost Risk Tier", loc="upper left")
ax1.grid(True, which="both", linestyle=":", alpha=0.5)
plt.tight_layout()
fig1_path = OUTPUT_DIR / "feature2_parity_outliers.png"
plt.savefig(fig1_path, dpi=300, bbox_inches="tight")
plt.show()

# Plot 2: Multivariate LOF Score vs Peer Cost Z-Score
fig2, ax2 = plt.subplots(figsize=(10, 6))
sns.scatterplot(
    data=plot_df,
    x="upper_cost_zscore",
    y="lof_anomaly_score",
    hue="cost_risk_tier",
    hue_order=["Low", "Medium", "High", "Critical"],
    palette={"Low": "#2ecc71", "Medium": "#f39c12", "High": "#e67e22", "Critical": "#e74c3c"},
    alpha=0.7,
    s=40,
    ax=ax2
)

ax2.axvline(2.5, color="orange", linestyle="--", alpha=0.7, label="Z = 2.5 (High Risk)")
ax2.axvline(4.0, color="red", linestyle="--", alpha=0.7, label="Z = 4.0 (Critical Risk)")
ax2.axhline(0.70, color="purple", linestyle=":", alpha=0.7, label="LOF Score = 0.70 (Multivariate Outlier)")

ax2.set_xlabel("Peer Cost Z-Score (Univariate Overrun)", fontsize=11)
ax2.set_ylabel("LOF Anomaly Score (Multivariate)", fontsize=11)
ax2.set_title("Feature 2: Multivariate LOF Score vs. Univariate Z-Score", fontsize=13, fontweight="bold")
ax2.legend(loc="lower right")
ax2.grid(True, alpha=0.4)
plt.tight_layout()
fig2_path = OUTPUT_DIR / "feature2_lof_vs_zscore.png"
plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.show()

# Plot 3: Category-Wise Outlier Distribution (Top Categories Strip Plot)
top_categories = plot_df["category_nlp_clean"].value_counts().head(5).index
cat_df = plot_df[plot_df["category_nlp_clean"].isin(top_categories)].copy()

fig3, ax3 = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=cat_df,
    y="category_nlp_clean",
    x="sanction_amount",
    color="#ecf0f1",
    fliersize=0,
    width=0.4,
    ax=ax3
)

sns.stripplot(
    data=cat_df,
    y="category_nlp_clean",
    x="sanction_amount",
    hue="cost_risk_tier",
    hue_order=["Low", "Medium", "High", "Critical"],
    palette={"Low": "#95a5a6", "Medium": "#f39c12", "High": "#e67e22", "Critical": "#e74c3c"},
    alpha=0.6,
    jitter=0.2,
    size=5,
    ax=ax3
)

ax3.set_xscale("log")
ax3.set_xlabel("Sanction Amount (Rs., Log Scale)", fontsize=11)
ax3.set_ylabel("Work Category (NLP Inferred)", fontsize=11)
ax3.set_title("Cost Outlier Distribution Across Top 5 Categories", fontsize=13, fontweight="bold")
ax3.legend(title="Risk Tier", loc="lower right")
ax3.grid(True, which="both", linestyle=":", alpha=0.4)
plt.tight_layout()
fig3_path = OUTPUT_DIR / "feature2_category_outliers.png"
plt.savefig(fig3_path, dpi=300, bbox_inches="tight")
plt.show()

logger.info("Generated and saved outlier visualization plots to %s, %s, %s", fig1_path, fig2_path, fig3_path)
